# 05 · Dates, timestamps, intervals and timezone semantics

**Scenario:** Derive duration, date boundaries and local-time display fields.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Derive date attributes

The taxi sample timestamps do not carry explicit per-record timezone metadata. Confirm the source convention before interpreting them as instants.


In [ ]:
print('Session timezone:',spark.conf.get('spark.sql.session.timeZone'))
dated=(df.withColumn('pickup_hour',F.hour('pickup_ts')).withColumn('month_start',F.trunc('pickup_date','month'))
    .withColumn('weekday',F.dayofweek('pickup_date')).withColumn('next_day',F.date_add('pickup_date',1)))
dated.select('pickup_ts','pickup_hour','month_start','weekday','duration_sec').show(8)
assert dated.filter((F.col('pickup_hour')<0)|(F.col('pickup_hour')>23)).count()==0


**Expected check:** Non-null hour values lie between 0 and 23.


## Step 3 — Round-trip formatting without guessing timezones

Formatting is presentation. Preserve typed timestamps for joins, ordering and interval calculations.


In [ ]:
roundtrip=(df.withColumn('timestamp_text',F.date_format('pickup_ts','yyyy-MM-dd HH:mm:ss'))
    .withColumn('parsed_ts',F.to_timestamp('timestamp_text','yyyy-MM-dd HH:mm:ss')))
roundtrip.select('pickup_ts','timestamp_text','parsed_ts').show(5,truncate=False)
assert roundtrip.filter(F.col('pickup_ts').isNotNull()&F.col('parsed_ts').isNull()).count()==0


**Expected check:** Formatted valid timestamps parse successfully in the current session timezone.


## Practice

Define an inclusive start date and exclusive end date for a job filter. Explain DST effects when converting local wall time to UTC.

## Solution guide

Use pickup_ts >= start and pickup_ts < end. Do not use a hardcoded 23:59:59 end. Only apply to_utc_timestamp after establishing the source timezone convention.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
